In [ ]:
import pandas as pd
!pip install lifetimes
import lifetimes as lf
from lifetimes.utils import summary_data_from_transaction_data
import numpy as np


In [ ]:
df = pd.read_csv('/content/scanner_data.csv')

In [ ]:
df

,Unnamed: 0,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
0,1,02/01/2016,2547,1,X52,0EM7L,1.0,3.13
1,2,02/01/2016,822,2,2ML,68BRQ,1.0,5.46
2,3,02/01/2016,3686,3,0H2,CZUZX,1.0,6.35
3,4,02/01/2016,3719,4,0H2,549KK,1.0,5.59
4,5,02/01/2016,9200,5,0H2,K8EHH,1.0,6.88
...,...,...,...,...,...,...,...,...
131701,131702,04/07/2016,20203,32900,IEV,FO112,3.0,6.46
131702,131703,04/07/2016,20203,32900,N8U,I36F2,1.0,4.50
131703,131704,04/07/2016,20203,32900,U5F,4X8P4,1.0,5.19
131704,131705,04/07/2016,20203,32900,0H2,ZVTO4,1.0,4.57


In [ ]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

In [ ]:
summary_1 = lf.utils.summary_data_from_transaction_data(
    df,
    customer_id_col='Customer_ID',
    datetime_col='Date',
    monetary_value_col='Sales_Amount'

)

In [ ]:
summary_1

,frequency,recency,T,monetary_value
Customer_ID,,,,
1,0.0,0.0,344.0,0.0000
2,1.0,87.0,282.0,15.0000
3,0.0,0.0,364.0,0.0000
4,1.0,121.0,173.0,26.6000
5,4.0,147.0,326.0,14.6725
...,...,...,...,...
22621,0.0,0.0,21.0,0.0000
22622,0.0,0.0,15.0,0.0000
22623,0.0,0.0,10.0,0.0000


In [ ]:
summary_1 = summary_1[summary_1['monetary_value'] > 0]

In [ ]:
summary_1

,frequency,recency,T,monetary_value
Customer_ID,,,,
2,1.0,87.0,282.0,15.000000
4,1.0,121.0,173.0,26.600000
5,4.0,147.0,326.0,14.672500
13,1.0,31.0,355.0,22.920000
14,3.0,59.0,347.0,102.033333
...,...,...,...,...
22562,1.0,2.0,12.0,14.940000
22580,1.0,1.0,8.0,5.370000
22605,1.0,5.0,17.0,12.350000


In [ ]:
bgf = lf.BetaGeoFitter(penalizer_coef=0.05)
bgf.fit(summary_1['frequency'], summary_1['recency'], summary_1['T'])

<lifetimes.BetaGeoFitter: fitted with 10766 subjects, a: 0.29, alpha: 40.36, b: 0.60, r: 1.23>

In [ ]:

summary_1['probability_alive'] = bgf.conditional_probability_alive(summary_1['frequency'], summary_1['recency'], summary_1['T'])

In [ ]:
summary_1 = summary_1.drop(columns=['probality_alive'])
summary_1

,frequency,recency,T,monetary_value,Customer_type,probability_alive
Customer_ID,,,,,,
2,1.0,87.0,282.0,15.000000,At Risk,0.208986
4,1.0,121.0,173.0,26.600000,Healthy,0.528776
5,4.0,147.0,326.0,14.672500,At Risk,0.273075
13,1.0,31.0,355.0,22.920000,At Risk,0.044106
14,3.0,59.0,347.0,102.033333,At Risk,0.027902
...,...,...,...,...,...,...
22562,1.0,2.0,12.0,14.940000,Healthy,0.565947
22580,1.0,1.0,8.0,5.370000,Healthy,0.596076
22605,1.0,5.0,17.0,12.350000,Healthy,0.553445


In [ ]:
summary_1['Customer_type'] = np.where(summary_1['probability_alive'] < 0.3,'At Risk','Healthy')

In [ ]:
ggf = lf.GammaGammaFitter(penalizer_coef=0.1)
ggf.fit(summary_1['frequency'], summary_1['monetary_value'])

<lifetimes.GammaGammaFitter: fitted with 10766 subjects, p: 1.27, q: 0.40, v: 1.13>

In [ ]:
summary_1['predicted_sale'] = ggf.customer_lifetime_value(bgf,
                                                          summary_1['frequency'],
                                                          summary_1['recency'],
                                                          summary_1['T'],
                                                          summary_1['monetary_value'],
                                                          time = 12,
                                                          freq = 'D',
                                                          discount_rate=0.01)

In [ ]:
summary_1

,frequency,recency,T,monetary_value,Customer_type,probability_alive,predicted_sale
Customer_ID,,,,,,,
2,1.0,87.0,282.0,15.000000,At Risk,0.208986,12.431614
4,1.0,121.0,173.0,26.600000,Healthy,0.528776,77.052710
5,4.0,147.0,326.0,14.672500,At Risk,0.273075,19.609350
13,1.0,31.0,355.0,22.920000,At Risk,0.044106,3.269879
14,3.0,59.0,347.0,102.033333,At Risk,0.027902,10.980386
...,...,...,...,...,...,...,...
22562,1.0,2.0,12.0,14.940000,Healthy,0.565947,148.619249
22580,1.0,1.0,8.0,5.370000,Healthy,0.596076,67.359882
22605,1.0,5.0,17.0,12.350000,Healthy,0.553445,113.579432


In [ ]:
summary_1['predicted_sale'].sum()

np.float64(1104357.668148472)

In [ ]:
top_customers = summary_1.sort_values(by='predicted_sale', ascending=False).head(20)

In [ ]:
top_customers

,frequency,recency,T,monetary_value,Customer_type,probability_alive,predicted_sale
Customer_ID,,,,,,,
12949,6.0,84.0,99.0,272.131667,Healthy,0.895194,3645.498968
17104,50.0,293.0,293.0,76.320200,Healthy,0.994228,3538.119464
17471,59.0,274.0,290.0,66.398814,Healthy,0.910981,3339.627368
17294,36.0,277.0,292.0,96.791944,Healthy,0.956811,3155.735300
15677,24.0,273.0,284.0,114.792083,Healthy,0.971690,2646.957075
15540,33.0,262.0,276.0,79.733636,Healthy,0.960075,2510.664916
17968,34.0,281.0,291.0,66.656176,Healthy,0.975397,2103.777069
20906,2.0,161.0,213.0,593.940000,Healthy,0.726067,2012.490196
18922,13.0,248.0,248.0,135.915385,Healthy,0.977659,2004.211390


In [ ]:
top_customers['predicted_sale'].head(20).sum()
profit_rate = 0.15
summary_1['predicted_profit'] = summary_1['predicted_sale'] * profit_rate
summary_1['predicted_sale'].mean()
customer_count = summary_1['Customer_type'].count()